# KuchoLM training

日本語コーパスを MeCab で NIDA_FICTION へ変換し、その JSONL から小型 seq2seq Transformer を Colab GPU で学習します。

この notebook だけで、データ生成 → 10件確認 → tokenizer 学習 → モデル学習 → 保存 → 推論まで通します。

In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece

## 1. 設定

In [ ]:
from pathlib import Path
import json, math, random, re
import MeCab
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_ROWS = 100_000
USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')
DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
tagger = MeCab.Tagger()

## 2. コーパス読み込み

In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.rstrip('\n') for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]) for row in dataset)

## 3. NIDA_FICTION 変換

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_SPLIT_RE = re.compile(r'(.+?[。！？!?]+|.+$)', re.S)
TRAILING_WS_RE = re.compile(r'(\s*)$')

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            f = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': f[0] if len(f) > 0 else '',
                'ctype': f[4] if len(f) > 4 else '*',
                'lemma': f[7] if len(f) > 7 else '*',
                'orth_base': f[10] if len(f) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    if '下一段' in ctype or '上一段' in ctype or '一段' in ctype:
        return True
    return base.endswith('る') and token.get('surface', '') == base[:-1]

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た'
    if base.endswith(('う', 'つ', 'る')): return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    mapping = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    last = base[-1:]
    return base[:-1] + mapping[last] + 'ない' if last in mapping else base + 'ない'

def soft_ending(body, is_question):
    if is_question: return 'ニカ'
    if re.search(r'(ね|よ|な)$', body): return 'ニダ'
    if re.search(r'(ない|ません|難しい|心配|残念|大丈夫)$', body): return 'ニダね'
    return 'ニダよ'

def split_sentences_preserve(text):
    parts = []
    cursor = 0
    for match in SENTENCE_SPLIT_RE.finditer(text):
        start, end = match.span()
        if start > cursor:
            parts.append((text[cursor:start], False))
        chunk = match.group(0)
        trailing = TRAILING_WS_RE.search(chunk).group(1)
        core = chunk[:-len(trailing)] if trailing else chunk
        if core:
            parts.append((core, True))
        if trailing:
            parts.append((trailing, False))
        cursor = end
    if cursor < len(text):
        parts.append((text[cursor:], False))
    return parts

def soften_surface(text):
    for pattern, replacement in [
        (r'ということです$', 'ってこと'),
        (r'ということでした$', 'ってことだった'),
        (r'のであります$', 'んだ'),
        (r'であります$', 'なんだ'),
        (r'なのです$', 'なんだ'),
        (r'のです$', 'んだ'),
        (r'でしょう$', 'だろう'),
        (r'ではありません$', 'じゃない'),
        (r'ではないです$', 'じゃない'),
        (r'ではない$', 'じゃない'),
    ]:
        text = re.sub(pattern, replacement, text)
    return text

def convert_polite_tail(body):
    tokens = parse_tokens(body)
    if not tokens:
        return body
    surfaces = [t['surface'] for t in tokens]
    patterns = [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]
    for suffix, mode in patterns:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix:
            continue
        suffix_start = len(tokens) - len(suffix)
        verb_index = next((i for i in range(suffix_start - 1, -1, -1) if tokens[i]['pos'] == '動詞'), None)
        if verb_index is None:
            continue
        verb = tokens[verb_index]
        base = dictionary_form(verb)
        prefix = ''.join(t['surface'] for t in tokens[:verb_index])

        aux_map = {
            '来る': {'present':'くる', 'past':'きた', 'negative':'こない', 'negative_past':'こなかった'},
            '行く': {'present':'いく', 'past':'いった', 'negative':'いかない', 'negative_past':'いかなかった'},
            '仕舞う': {'present':'しまう', 'past':'しまった', 'negative':'しまわない', 'negative_past':'しまわなかった'},
            'しまう': {'present':'しまう', 'past':'しまった', 'negative':'しまわない', 'negative_past':'しまわなかった'},
        }
        if base in aux_map and verb_index > 0 and tokens[verb_index - 1]['surface'] in {'て', 'で'}:
            stem = ''.join(t['surface'] for t in tokens[:verb_index])
            return stem + aux_map[base][mode]

        if mode == 'present': replacement = base
        elif mode == 'past': replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'
        return prefix + replacement

    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だった'
    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1])
    return ''.join(surfaces)

def convert_sentence(sentence):
    leading = re.match(r'^\s*', sentence).group(0)
    trailing = re.search(r'\s*$', sentence).group(0)
    core = sentence[len(leading): len(sentence) - len(trailing) if trailing else None]
    if not core or URL_RE.search(core): return sentence

    punct_match = re.search(r'([。！？!?]+)$', core)
    punctuation = punct_match.group(1) if punct_match else ''
    body = core[:-len(punctuation)] if punctuation else core
    is_question = bool(re.search(r'[？?]$', punctuation))
    body = soften_surface(body)

    if body.endswith('か') and is_question:
        body = body[:-1]
    converted = convert_polite_tail(body)

    if converted.endswith(('ね', 'よ', 'な')):
        particle = converted[-1]
        converted = converted[:-1] + 'ニダ' + particle
    else:
        converted += soft_ending(body, is_question)

    return leading + converted + (punctuation or ('？' if is_question else '')) + trailing

def to_nida(text):
    if not text or URL_RE.search(text):
        return None
    out = []
    for part, is_sentence in split_sentences_preserve(text):
        out.append(convert_sentence(part) if is_sentence else part)
    result = ''.join(out)
    result = re.sub(r'(ました|ません)(ニダ)', r'\1', result)
    return result


## 4. JSONL生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        if not source or len(source.strip()) < 2 or len(source) > 256:
            continue
        target = to_nida(source)
        if not target or target == source:
            continue
        output.write(json.dumps({'style':'NIDA_FICTION','source':source,'target':target}, ensure_ascii=False) + '\n')
        written += 1
        if written >= MAX_ROWS:
            break
print('written:', written)
print('size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)

## 5. 生成サンプルを10件確認

In [ ]:
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        item = json.loads(line)
        print(f"{i + 1:02d}. {item['source']} -> {item['target']}")

## 6. 品質チェック

In [ ]:
checks = {
    'space_lost': 0,
    'polite_left': 0,
    'broken_aux': 0,
}
samples = {k: [] for k in checks}
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        source, target = item['source'], item['target']
        flags = {
            'space_lost': source.count(' ') != target.count(' '),
            'polite_left': bool(re.search(r'(ました|ません)ニダ', target)),
            'broken_aux': bool(re.search(r'(くった|いったった|しまったった)ニダ', target)),
        }
        for key, hit in flags.items():
            if hit:
                checks[key] += 1
                if len(samples[key]) < 5:
                    samples[key].append((source, target))
print(checks)
for key, rows in samples.items():
    print('\n', key)
    for source, target in rows:
        print(source, '->', target)

## 7. 学習データ読込

In [ ]:
rows = []
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        rows.append((f"<NIDA_FICTION> {item['source']}", item['target']))
random.shuffle(rows)
split = int(len(rows) * 0.98)
train_rows, val_rows = rows[:split], rows[split:]
print('train:', len(train_rows), 'val:', len(val_rows))

## 8. SentencePiece tokenizer

In [ ]:
VOCAB_SIZE = 8000
spm_corpus = WORK_DIR / 'spm_corpus.txt'
with spm_corpus.open('w', encoding='utf-8') as f:
    for source, target in rows:
        f.write(source + '\n' + target + '\n')
spm.SentencePieceTrainer.train(input=str(spm_corpus), model_prefix=str(WORK_DIR/'kucholm_spm'), vocab_size=VOCAB_SIZE, model_type='bpe', character_coverage=0.9995, pad_id=0, unk_id=1, bos_id=2, eos_id=3)
sp = spm.SentencePieceProcessor(model_file=str(WORK_DIR/'kucholm_spm.model'))

## 9. Dataset / Model / Training

In [ ]:
MAX_LEN=128; D_MODEL=256; NHEAD=8; ENC_LAYERS=4; DEC_LAYERS=4; FF=1024; BATCH=64 if device.type=='cuda' else 8; EPOCHS=3; LR=3e-4
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
def encode(text): return [BOS_ID, *sp.encode(text, out_type=int)[:MAX_LEN-2], EOS_ID]
class DS(Dataset):
    def __init__(self, items): self.items=items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        s,t=self.items[i]; return torch.tensor(encode(s)), torch.tensor(encode(t))
def collate(batch):
    s,t=zip(*batch)
    return nn.utils.rnn.pad_sequence(s,batch_first=True,padding_value=PAD_ID), nn.utils.rnn.pad_sequence(t,batch_first=True,padding_value=PAD_ID)
train_loader=DataLoader(DS(train_rows),batch_size=BATCH,shuffle=True,collate_fn=collate)
val_loader=DataLoader(DS(val_rows),batch_size=BATCH,shuffle=False,collate_fn=collate)
class Pos(nn.Module):
    def __init__(self,d,max_len=2048):
        super().__init__(); p=torch.arange(max_len).unsqueeze(1); div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.0)/d)); pe=torch.zeros(max_len,d); pe[:,0::2]=torch.sin(p*div); pe[:,1::2]=torch.cos(p*div); self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]
class Model(nn.Module):
    def __init__(self):
        super().__init__(); self.emb=nn.Embedding(sp.vocab_size(),D_MODEL,padding_idx=PAD_ID); self.pos=Pos(D_MODEL); self.tr=nn.Transformer(d_model=D_MODEL,nhead=NHEAD,num_encoder_layers=ENC_LAYERS,num_decoder_layers=DEC_LAYERS,dim_feedforward=FF,dropout=.1,batch_first=True); self.head=nn.Linear(D_MODEL,sp.vocab_size(),bias=False); self.head.weight=self.emb.weight
    def forward(self,src,tgt):
        sm=src.eq(PAD_ID); tm=tgt.eq(PAD_ID); mask=nn.Transformer.generate_square_subsequent_mask(tgt.size(1),device=tgt.device); s=self.pos(self.emb(src)*math.sqrt(D_MODEL)); t=self.pos(self.emb(tgt)*math.sqrt(D_MODEL)); return self.head(self.tr(s,t,tgt_mask=mask,src_key_padding_mask=sm,tgt_key_padding_mask=tm,memory_key_padding_mask=sm))
model=Model().to(device); print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M params')
opt=torch.optim.AdamW(model.parameters(),lr=LR); loss_fn=nn.CrossEntropyLoss(ignore_index=PAD_ID); scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
for epoch in range(1,EPOCHS+1):
    model.train(); total=0
    for src,tgt in train_loader:
        src,tgt=src.to(device),tgt.to(device); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'):
            logits=model(src,tgt[:,:-1]); loss=loss_fn(logits.reshape(-1,logits.size(-1)),tgt[:,1:].reshape(-1))
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); total+=loss.item()
    print(f'epoch {epoch}: {total/max(1,len(train_loader)):.4f}')
    torch.save(model.state_dict(), WORK_DIR/f'KuchoLM-NIDA-epoch{epoch}.pt')

## 10. 保存 / 推論

In [ ]:
torch.save({'model':model.state_dict(),'config':{'vocab_size':sp.vocab_size(),'d_model':D_MODEL,'nhead':NHEAD,'enc_layers':ENC_LAYERS,'dec_layers':DEC_LAYERS,'ff':FF}}, WORK_DIR/'KuchoLM-NIDA.pt')
@torch.no_grad()
def infer(text,max_new=96):
    model.eval(); src=torch.tensor([encode('<NIDA_FICTION> '+text)],device=device); out=torch.tensor([[BOS_ID]],device=device)
    for _ in range(max_new):
        nxt=model(src,out)[:,-1].argmax(-1,keepdim=True); out=torch.cat([out,nxt],1)
        if nxt.item()==EOS_ID: break
    return sp.decode([i for i in out[0].tolist() if i not in {PAD_ID,BOS_ID,EOS_ID}])
for x in ['今日は学校です。','魚を食べました。','最近ちょっと疲れています。']:
    print(x,'->',infer(x))